# R03a - Decoder auxiliar

Pregunta objetivo: ?el encoder de R03 sigue guardando textura en `mu`, o el decoder original ya no es capaz de recuperarla?

Este notebook reutiliza la ruta oficial del proyecto para construir el modelo y cargar checkpoints (`build_vae(...) + load_checkpoint(...)`) y entrena un decoder auxiliar de mayor capacidad sobre `mu` congelado.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm.auto import tqdm

from poregen.experiments.r03 import (
    AuxiliaryXCTDecoder,
    build_patch_loader,
    evaluate_auxiliary_decoder,
    find_repo_root,
    find_transition_focus,
    load_r03_runtime,
    mask_local_variance_density,
    sigmoid_xct,
    train_auxiliary_decoder,
    transition_percentile_threshold,
)
from poregen.training import seed_everything


REPO_ROOT = find_repo_root()
CHECKPOINT_PATH = REPO_ROOT / "runs" / "vae" / "r03" / "last.ckpt"
CONFIG_PATH = REPO_ROOT / "src" / "poregen" / "configs" / "r03.yaml"
DATA_ROOT = None
OUTPUT_DIR = REPO_ROOT / "runs" / "analysis" / "r03a_auxiliary_decoder"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRANSITION_KERNEL_SIZE = 5
AUX_BATCH_SIZE = 64
AUX_LR = 3e-4
AUX_WEIGHT_DECAY = 1e-2
AUX_MAX_EPOCHS = 8
AUX_PATIENCE = 2
FORCE_RETRAIN_AUX = False
PLOT_CASES_PER_CLASS = 3
ZOOM_RADIUS = 12

seed_everything(42)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}. Update CHECKPOINT_PATH once the R03 run finishes."
    )


In [ ]:
runtime = load_r03_runtime(
    CHECKPOINT_PATH,
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT,
    repo_root=REPO_ROOT,
)

AUX_BATCH_SIZE = min(AUX_BATCH_SIZE, int(runtime.cfg["data"].get("batch_size", AUX_BATCH_SIZE)))

threshold_loader = build_patch_loader(
    runtime.cfg,
    runtime.data_root,
    "train",
    batch_size=AUX_BATCH_SIZE,
    shuffle=False,
)
train_loader = build_patch_loader(
    runtime.cfg,
    runtime.data_root,
    "train",
    batch_size=AUX_BATCH_SIZE,
    shuffle=True,
)
val_loader = build_patch_loader(runtime.cfg, runtime.data_root, "val", batch_size=AUX_BATCH_SIZE, shuffle=False)
test_loader = build_patch_loader(runtime.cfg, runtime.data_root, "test", batch_size=AUX_BATCH_SIZE, shuffle=False)

print(f"Loaded R03 checkpoint at step {runtime.checkpoint_step} on {runtime.device}.")
print(f"Data root: {runtime.data_root}")
print(f"Train / val / test patches: {len(train_loader.dataset):,} / {len(val_loader.dataset):,} / {len(test_loader.dataset):,}")


In [ ]:
def collect_transition_scores(loader, kernel_size: int, desc: str) -> torch.Tensor:
    scores = []
    for batch in tqdm(loader, desc=desc):
        scores.append(mask_local_variance_density(batch["mask"], kernel_size=kernel_size).cpu())
    return torch.cat(scores)


train_transition_scores = collect_transition_scores(
    threshold_loader,
    kernel_size=TRANSITION_KERNEL_SIZE,
    desc="Train transition density",
)
transition_threshold = transition_percentile_threshold(train_transition_scores, percentile=75.0)
print(f"75th percentile transition-density threshold: {transition_threshold:.6f}")
pd.Series(train_transition_scores.numpy()).describe(percentiles=[0.25, 0.5, 0.75, 0.9])


In [ ]:
aux_checkpoint = OUTPUT_DIR / "aux_decoder.pt"
history_path = OUTPUT_DIR / "aux_history.csv"

if aux_checkpoint.exists() and not FORCE_RETRAIN_AUX:
    aux_decoder = AuxiliaryXCTDecoder(runtime.model.cfg).to(runtime.device)
    aux_decoder.load_state_dict(torch.load(aux_checkpoint, map_location=runtime.device))
    aux_decoder.eval()
    history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
else:
    aux_decoder, history = train_auxiliary_decoder(
        runtime.model,
        train_loader,
        val_loader,
        runtime.device,
        lr=AUX_LR,
        weight_decay=AUX_WEIGHT_DECAY,
        max_epochs=AUX_MAX_EPOCHS,
        patience=AUX_PATIENCE,
    )
    torch.save(aux_decoder.state_dict(), aux_checkpoint)
    history.to_csv(history_path, index=False)

history


In [ ]:
val_metrics = evaluate_auxiliary_decoder(
    runtime.model,
    aux_decoder,
    val_loader,
    runtime.device,
    split_name="val",
    transition_threshold=transition_threshold,
    transition_kernel_size=TRANSITION_KERNEL_SIZE,
)
test_metrics = evaluate_auxiliary_decoder(
    runtime.model,
    aux_decoder,
    test_loader,
    runtime.device,
    split_name="test",
    transition_threshold=transition_threshold,
    transition_kernel_size=TRANSITION_KERNEL_SIZE,
)

metrics = pd.concat([val_metrics, test_metrics], ignore_index=True)
metrics["xct_loss_gain"] = metrics["xct_loss_original"] - metrics["xct_loss_auxiliary"]
metrics["sharpness_ratio_abs_error_original"] = (metrics["sharpness_ratio_original"] - 1.0).abs()
metrics["sharpness_ratio_abs_error_auxiliary"] = (metrics["sharpness_ratio_auxiliary"] - 1.0).abs()
metrics.to_parquet(OUTPUT_DIR / "patch_metrics.parquet", index=False)

summary = (
    metrics.groupby(["split", "transition_label"], as_index=False)
    .agg(
        n_patches=("dataset_index", "count"),
        xct_loss_original=("xct_loss_original", "mean"),
        xct_loss_auxiliary=("xct_loss_auxiliary", "mean"),
        xct_loss_gain=("xct_loss_gain", "mean"),
        sharpness_recon_original=("sharpness_recon_original", "mean"),
        sharpness_recon_auxiliary=("sharpness_recon_auxiliary", "mean"),
        sharpness_ratio_original=("sharpness_ratio_original", "mean"),
        sharpness_ratio_auxiliary=("sharpness_ratio_auxiliary", "mean"),
        sharpness_ratio_abs_error_original=("sharpness_ratio_abs_error_original", "mean"),
        sharpness_ratio_abs_error_auxiliary=("sharpness_ratio_abs_error_auxiliary", "mean"),
    )
)
summary


In [ ]:
datasets = {
    "val": val_loader.dataset,
    "test": test_loader.dataset,
}


def crop_2d(image: torch.Tensor, center_y: int, center_x: int, radius: int = ZOOM_RADIUS) -> torch.Tensor:
    y0 = max(center_y - radius, 0)
    y1 = min(center_y + radius, image.shape[0])
    x0 = max(center_x - radius, 0)
    x1 = min(center_x + radius, image.shape[1])
    return image[y0:y1, x0:x1]


def fetch_case(row: pd.Series) -> dict[str, object]:
    sample = datasets[row["split"]][int(row["dataset_index"])]
    xct = sample["xct"].unsqueeze(0).to(runtime.device)
    mask = sample["mask"].unsqueeze(0).to(runtime.device)
    with torch.no_grad():
        output = runtime.model(xct, mask)
        aux_logits = aux_decoder(output.mu)

    focus_z, focus_y, focus_x = find_transition_focus(sample["mask"], kernel_size=TRANSITION_KERNEL_SIZE)
    return {
        "xct": sample["xct"][0].cpu(),
        "original": sigmoid_xct(output.xct_logits)[0, 0].cpu(),
        "auxiliary": sigmoid_xct(aux_logits)[0, 0].cpu(),
        "focus": (focus_z, focus_y, focus_x),
    }


def plot_case(row: pd.Series) -> None:
    case = fetch_case(row)
    focus_z, focus_y, focus_x = case["focus"]
    full_images = [
        case["xct"][focus_z],
        case["original"][focus_z],
        case["auxiliary"][focus_z],
    ]
    zoom_images = [crop_2d(image, focus_y, focus_x) for image in full_images]
    titles = ["XCT original", "R03", "Auxiliar"]

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, image, title in zip(axes[0], full_images, titles):
        ax.imshow(image, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis("off")
    for ax, image, title in zip(axes[1], zoom_images, titles):
        ax.imshow(image, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{title} (zoom)")
        ax.axis("off")

    fig.suptitle(
        f"{row['split']} idx={int(row['dataset_index'])} | {row['transition_label']} | loss gain={row['xct_loss_gain']:.5f}"
    )
    plt.tight_layout()
    plt.show()


representatives = (
    pd.concat(
        [
            metrics.query("transition_label == 'high_transition'").nlargest(PLOT_CASES_PER_CLASS, "xct_loss_gain"),
            metrics.query("transition_label == 'interior'").nlargest(PLOT_CASES_PER_CLASS, "xct_loss_gain"),
        ],
        ignore_index=True,
    )
    .drop_duplicates(subset=["split", "dataset_index"])
)
display(representatives[[
    "split",
    "dataset_index",
    "transition_label",
    "xct_loss_gain",
    "sharpness_ratio_original",
    "sharpness_ratio_auxiliary",
]])

for _, representative_row in representatives.iterrows():
    plot_case(representative_row)


In [ ]:
transition_mean_gain = summary.loc[summary["transition_label"] == "high_transition", "xct_loss_gain"].mean()
interior_mean_gain = summary.loc[summary["transition_label"] == "interior", "xct_loss_gain"].mean()
transition_ratio_error_drop = (
    summary.loc[summary["transition_label"] == "high_transition", "sharpness_ratio_abs_error_original"].mean()
    - summary.loc[summary["transition_label"] == "high_transition", "sharpness_ratio_abs_error_auxiliary"].mean()
)

print(f"Mean XCT loss gain in high-transition patches: {transition_mean_gain:.6f}")
print(f"Mean XCT loss gain in interior patches: {interior_mean_gain:.6f}")
print(f"Drop in |sharpness_ratio - 1| for high-transition patches: {transition_ratio_error_drop:.6f}")

if transition_mean_gain > max(interior_mean_gain * 1.25, 0.0):
    print("Interpretation: the auxiliary decoder is materially better in transition-rich regions, so the main bottleneck is likely decoder capacity / decoder supervision.")
    print("Suggested R04 direction: keep z similar, add decoder-side texture pressure (2-D perceptual loss, adversarial loss, or both).")
else:
    print("Interpretation: the auxiliary decoder does not buy much over R03, so mu may already be losing the texture information.")
    print("Suggested R04 direction: increase latent bandwidth (z_channels) and/or reduce the downsampling factor f.")
